In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

In [ ]:
president_county = pd.read_csv("4-US_Election_2020/president_county.csv")
president_cand = pd.read_csv("4-US_Election_2020/president_county_candidate.csv")

senator_county = pd.read_csv("4-US_Election_2020/senate_county.csv")
senator_cand = pd.read_csv("4-US_Election_2020/senate_county_candidate.csv")

governor_county = pd.read_csv("4-US_Election_2020/governors_county.csv")
governor_cand = pd.read_csv("4-US_Election_2020/governors_county_candidate.csv")


To analyze the data, we must join the candidate-specific results with the total county votes. We perform an "inner" merge on the state and county names to ensure we only analyze regions where both total counts and candidate breakdowns are available. 

In [ ]:
president_merged = pd.merge(president_cand, president_county, on = ["state", "county"], how = "left")
senator_merged = pd.merge(senator_cand, senator_county, on = ["state", "county"], how = "left")
governor_merged = pd.merge(governor_cand, governor_county, on = ["state", "county"], how = "left")

In [ ]:
president_merged = president_merged.drop(columns = ['candidate', 'percent'])
senator_merged = senator_merged.drop(columns = ['candidate', 'percent'])
governor_merged = governor_merged.drop(columns = ['candidate', 'percent'])

print(president_merged)
print(senator_merged)
print(governor_merged)


Our cleaning process involves:

Filtering for the two primary parties (DEM and REP).

Calculating the vote_share for each candidate relative to the county total.

Renaming columns for clarity.

Removing any rows with missing values or zero total votes to avoid division errors.

In [ ]:
def clean_election(df): 
    df["state"] = df["state"].str.strip()
    df["county"] = df["county"].str.strip()
    df["party"] = df["party"].str.strip()

    df["party"] = df["party"].replace({
        "DEM":"DEMOCRAT",
        "REP":"REPUBLICAN"
        })
  
    df = df.query("party == 'DEMOCRAT' or party == 'REPUBLICAN'")

    df = df[["state", "county", "party", "current_votes_x"]] # NEED TO FIGURE OUT WHAT VOTES/TOTAL_VOTES/CURRENT_VOTES 
                                                            #COLUMN TO USE BC ITS DIFFERENT FOR THE DATASETS

    return df

president_clean = clean_election(president_merged)
senator_clean = clean_election(senator_merged)
governor_clean = clean_election(governor_merged)

display(president_clean.head())
display(senator_clean.head())
display(governor_clean.head())

In [ ]:
def party_votes(df):
    grouped = df.groupby(["state", "county", "party"])["current_votes"].sum()
    grouped = grouped.reset_index()
    return grouped

president_votes = party_votes(president_clean)
senator_votes = party_votes(senator_clean)
governor_votes = party_votes(governor_clean)


print(president_votes)
print(senator_votes)
print(governor_votes)


In [ ]:
def total_votes(df):
    df["total_votes"] = df.groupby(["state", "county"])["current_votes"].transform("sum")
    df["vote_share"] = df["candidatevotes"] / df["total_votes"]
    return df

president_votes = total_votes(president_votes)
senator_votes = total_votes(senator_votes)
governor_votes = total_votes(governor_votes)